# 🔍 Lesson 02 — Oracle 23ai Vector Search
## Student Activity: Search a Wikipedia Article

In this activity you will:
1. **Choose** a Wikipedia article as your dataset
2. **Generate** vector embeddings for each paragraph (the "pre-load phase")
3. **Load** the vectors into Oracle 23ai on freesql.com
4. **Search** the article using natural language — no keyword matching needed

> This is exactly how Netflix, Spotify, and AI assistants find relevant content at scale.

## Step 1 — Install & Import

In [2]:
!pip install sentence-transformers wikipedia-api -q

from sentence_transformers import SentenceTransformer
import wikipediaapi
import re
import textwrap

print("Libraries loaded ✓")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 8.0 MB/s eta 0:00:00
Libraries loaded ✓


## Step 2 — Choose Your Article

Pick one of the three articles below by setting `ARTICLE_CHOICE` to 1, 2, or 3:

| # | Article | Focus |
|---|---------|-------|
| 1 | Vector database | What they are, how they're built |
| 2 | Word embedding | How words become numbers |
| 3 | Semantic search | How meaning-based search works |

In [3]:
# ── CHANGE THIS: pick 1, 2, or 3 ──
ARTICLE_CHOICE = 1

ARTICLES = {
    1: "Cow Tipping",
    2: "Word embedding",
    3: "Semantic search",
}

ARTICLE_TITLE = ARTICLES[ARTICLE_CHOICE]
print(f"You chose: '{"Cow Tipping"}'")

You chose: 'Cow Tipping'


## Step 3 — Fetch & Chunk the Article

In [4]:
wiki = wikipediaapi.Wikipedia(
    user_agent="oracle-vector-search-lesson/1.0",
    language="en"
)

page = wiki.page(ARTICLE_TITLE)
if not page.exists():
    raise ValueError(f"Article '{ARTICLE_TITLE}' not found on Wikipedia")

print(f"✓ Fetched: {page.title}")
print(f"  Length: {len(page.text):,} characters")

# Split into paragraphs, filter short/empty ones
raw_paragraphs = [p.strip() for p in page.text.split('\n') if len(p.strip()) > 120]

# Truncate to 400 chars max per chunk (fits Oracle VARCHAR2(2000) safely)
chunks = []
for i, para in enumerate(raw_paragraphs[:30]):   # cap at 30 chunks for this lesson
    chunk = para[:400]
    # Remove references like [1], [23]
    chunk = re.sub(r'\[\d+\]', '', chunk).strip()
    if len(chunk) > 80:
        chunks.append(chunk)

print(f"  Chunks: {len(chunks)}")
print()
print("Preview of first 3 chunks:")
for i, c in enumerate(chunks[:3]):
    print(f"  [{i+1}] {c[:100]}...")

✓ Fetched: Cow tipping
  Length: 11,823 characters
  Chunks: 22

Preview of first 3 chunks:
  [1] Cow tipping is the purported activity of sneaking up on any unsuspecting or sleeping upright cow and...
  [2] Cows routinely lie down and can easily regain their footing unless sick or injured. Scientific studi...
  [3] Some versions of the urban legend suggest that because cows sleep standing up, it is possible to app...


## Step 4 — Generate Vector Embeddings

In [5]:
# Load the model (downloads ~90MB on first run)
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded ✓")

# Encode all chunks
embeddings = model.encode(chunks, show_progress_bar=True)
print(f"\nGenerated {len(embeddings)} embeddings, each with {len(embeddings[0])} dimensions")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded ✓


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Generated 22 embeddings, each with 384 dimensions


## Step 5 — Set Up the Database

📋 **Copy the SQL below and run it in freesql.com** (only once — it creates the table)

> freesql.com → SQL Workshop → SQL Commands → paste → Run

In [6]:
setup_sql = """-- ============================================================
-- Lesson 02 Step 5: Create the doc_chunks table
-- Run this in freesql.com BEFORE loading data
-- ============================================================

-- Drop if it already exists from a previous run
BEGIN
  EXECUTE IMMEDIATE 'DROP TABLE doc_chunks';
EXCEPTION WHEN OTHERS THEN NULL;
END;
/

CREATE TABLE doc_chunks (
    chunk_id     NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    doc_name     VARCHAR2(200),
    chunk_text   VARCHAR2(2000),
    chunk_vector VECTOR(384, FLOAT32)
);

-- Verify
SELECT table_name FROM user_tables WHERE table_name = 'DOC_CHUNKS';
"""

print(setup_sql)
print("=" * 60)
print("📋 Copy everything above and run it in freesql.com")

-- ============================================================
-- Lesson 02 Step 5: Create the doc_chunks table
-- Run this in freesql.com BEFORE loading data
-- ============================================================

-- Drop if it already exists from a previous run
BEGIN
  EXECUTE IMMEDIATE 'DROP TABLE doc_chunks';
EXCEPTION WHEN OTHERS THEN NULL;
END;
/

CREATE TABLE doc_chunks (
    chunk_id     NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    doc_name     VARCHAR2(200),
    chunk_text   VARCHAR2(2000),
    chunk_vector VECTOR(384, FLOAT32)
);

-- Verify
SELECT table_name FROM user_tables WHERE table_name = 'DOC_CHUNKS';

📋 Copy everything above and run it in freesql.com


## Step 6 — Generate INSERT Statements

Run this cell → copy the output → paste into freesql.com

> **Why the split trick?** Oracle rejects string literals over 4000 chars at parse time.
> A 384-dimension vector is ~4600 chars, so we split it in half using `TO_CLOB() || ...`

In [7]:
def format_vector(embedding):
    """Format embedding as Oracle VECTOR literal, split to avoid ORA-01704."""
    values = ", ".join(f"{v:.8f}" for v in embedding)
    full = f"[{values}]"
    mid = len(full) // 2
    split_pos = full.rindex(',', 0, mid) + 1
    part1 = full[:split_pos]
    part2 = full[split_pos:]
    return f"TO_VECTOR(TO_CLOB('{part1}') || '{part2}', 384, FLOAT32)"

print("-- ============================================================")
print(f"-- Lesson 02: Vector embeddings from '{ARTICLE_TITLE}'")
print("-- Run in freesql.com AFTER Step 5 (table must exist)")
print("-- ============================================================")
print()
for chunk, embedding in zip(chunks, embeddings):
    safe_text = chunk.replace("'", "''")[:390]   # stay under VARCHAR2(2000)
    vec = format_vector(embedding)
    print(f"INSERT INTO doc_chunks (doc_name, chunk_text, chunk_vector)")
    print(f"VALUES ('{ARTICLE_TITLE[:50]}', '{safe_text}', {vec});")
    print()
print("COMMIT;")
print()
print(f"-- Verify: {len(chunks)} rows expected")
print("SELECT COUNT(*) FROM doc_chunks;")

-- ============================================================
-- Lesson 02: Vector embeddings from 'Cow Tipping'
-- Run in freesql.com AFTER Step 5 (table must exist)
-- ============================================================

INSERT INTO doc_chunks (doc_name, chunk_text, chunk_vector)
VALUES ('Cow Tipping', 'Cow tipping is the purported activity of sneaking up on any unsuspecting or sleeping upright cow and pushing it over for entertainment. The practice of cow tipping is generally considered an urban legend and stories of such feats are viewed as tall tales. The implication that rural citizens seek such entertainment due to lack of alternatives is viewed as a stereotype.  The concept of cow', TO_VECTOR(TO_CLOB('[-0.00455768, -0.05851079, 0.00753304, 0.06157094, -0.01054936, -0.01785504, 0.09263958, 0.06219651, -0.00667455, 0.01563569, 0.09869090, -0.02178487, -0.00413213, 0.00462712, 0.00627578, -0.02329451, 0.01016348, -0.02699255, 0.07608223, 0.05810725, -0.00656324, -0.0442

## Step 7 — Search Your Article

Change `MY_QUESTION` below, run the cell, copy the SQL, paste it into freesql.com.

In [12]:
MY_QUESTION = "What is the philosophy?"
TOP_N = 3

q_emb = model.encode([MY_QUESTION])[0]
q_vec = format_vector(q_emb)

print(f"-- Search query: {MY_QUESTION}")
print(f"-- Top {TOP_N} most similar chunks")
print()
print("SELECT")
print("    chunk_id,")
print("    SUBSTR(chunk_text, 1, 100) AS preview,")
print(f"    ROUND(VECTOR_DISTANCE(chunk_vector, {q_vec}, COSINE), 4) AS similarity_score")
print("FROM doc_chunks")
print("ORDER BY similarity_score ASC")
print(f"FETCH FIRST {TOP_N} ROWS ONLY;")

-- Search query: What is the philosophy?
-- Top 3 most similar chunks

SELECT
    chunk_id,
    SUBSTR(chunk_text, 1, 100) AS preview,
    ROUND(VECTOR_DISTANCE(chunk_vector, TO_VECTOR(TO_CLOB('[-0.00629227, 0.09095301, -0.17564017, 0.00520874, -0.05592721, 0.00709165, 0.03270385, -0.04585378, 0.04669493, 0.01791062, 0.02728367, 0.04690357, -0.02648899, -0.03035539, -0.02419689, 0.00152668, -0.02853502, -0.05770085, -0.01539861, -0.00276594, -0.09041170, 0.03841561, -0.00523080, 0.03421461, -0.11310393, 0.04371321, 0.05818912, -0.04824411, 0.01593771, -0.04869674, 0.00610137, 0.10094339, 0.04005376, -0.04048568, -0.03910765, 0.04552872, 0.05282873, 0.04943374, 0.05141112, 0.02271718, 0.00455367, -0.02986817, -0.03859717, -0.01346590, -0.04090161, 0.00660809, -0.01866923, 0.02391479, 0.04566769, -0.03044582, -0.02171607, 0.00076020, -0.11094399, 0.08834823, 0.01552204, 0.06184974, -0.03290066, 0.00986718, -0.05935621, -0.08421945, 0.04721098, -0.02128319, -0.03896274, 0.15197358, 0.0591

## 🎯 Activity — Your Turn

Try these three searches. For each one, run Step 7 with a new question, paste the SQL in freesql.com, and write down what you found.

---

**Search 1:** Ask something that IS in the article
> Question: `"How much force would requiere cow tipping?"`

What came back? Does it make sense?



---

**Search 2:** Ask something that is RELATED but not a direct quote
> Example: `"What is the main reason of the idea that cows sleep standing up is considered a myth?"`

Did it find relevant content even though those exact words aren't in the article?

---

**Search 3:** Ask something UNRELATED
> Example: `"What is the philosophy?"`

What score did you get? Is it high or low? Why?

---

> 💡 **Key insight:** Vector search finds *meaning*, not keywords.
> A score near **0.0** = very similar. A score near **1.0** = very different.